## **Data Profiling**

In [47]:
import pandas as pd
import numpy as np

#### Load Datasets

In [48]:
years = [2011, 2012, 2014, 2018, 2021, 2023, 2024]

datasets = {}

for year in years:
    datasets[year] = pd.read_csv(
        f"../data/sparcs_{year}_raw.csv",
        low_memory=False
    )

#### Summary

In [49]:
summary = []

for year, df in datasets.items():
    summary.append({
        "Year": year,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

,Year,Rows,Columns,Memory (MB)
0,2011,2589121,33,3956.70
1,2012,2544543,33,3925.74
2,2014,2367550,33,3648.82
3,2018,2352807,33,3764.00
4,2021,2135260,33,3533.54
5,2023,2125754,33,3738.65
6,2024,2196737,33,3620.24


#### Schema Comparison & Validation

In [50]:
reference_columns = datasets[2024].columns

for year, df in datasets.items():

    print(f"\n{'='*60}")
    print(f"{year}")
    print("="*60)

    if reference_columns.equals(df.columns):
        print("✅ Column names and order match 2024")
    else:
        print("❌ Schema mismatch")


2011
✅ Column names and order match 2024

2012
✅ Column names and order match 2024

2014
✅ Column names and order match 2024

2018
✅ Column names and order match 2024

2021
✅ Column names and order match 2024

2023
✅ Column names and order match 2024

2024
✅ Column names and order match 2024


In [51]:
reference = datasets[2024].columns.tolist()

for year, df in datasets.items():

    if year == 2024:
        continue

    cols = df.columns.tolist()

    print(f"\n{'='*70}")
    print(f"{year}")
    print("="*70)

    extra = sorted(set(cols) - set(reference))
    missing = sorted(set(reference) - set(cols))

    print("Extra Columns:")
    print(extra if extra else "None")

    print("\nMissing Columns:")
    print(missing if missing else "None")


2011
Extra Columns:
None

Missing Columns:
None

2012
Extra Columns:
None

Missing Columns:
None

2014
Extra Columns:
None

Missing Columns:
None

2018
Extra Columns:
None

Missing Columns:
None

2021
Extra Columns:
None

Missing Columns:
None

2023
Extra Columns:
None

Missing Columns:
None


In [52]:
# -----------------------------
# Standardize All Datasets
# -----------------------------

rename_map = {
    "Facility ID": "Permanent Facility Id",
    "Hospital Service Area": "Health Service Area",
    "Zip Code - 3 digits": "Zip Code",

    "CCS Diagnosis Code": "CCSR Diagnosis Code",
    "CCS Diagnosis Description": "CCSR Diagnosis Description",

    "CCS Procedure Code": "CCSR Procedure Code",
    "CCS Procedure Description": "CCSR Procedure Description",

    "Source of Payment 1": "Payment Typology 1",
    "Source of Payment 2": "Payment Typology 2",
    "Source of Payment 3": "Payment Typology 3"
}

for year, df in datasets.items():

    # Rename columns
    df.rename(columns=rename_map, inplace=True)

    # Drop legacy column if present
    df.drop(columns=["Abortion Edit Indicator"], errors="ignore", inplace=True)
    df.to_csv(f"../data/sparcs_{year}_raw.csv", index=False)

In [53]:
reference_columns = datasets[2024].columns

for year, df in datasets.items():

    print(f"\n{'='*60}")
    print(year)
    print("="*60)

    if reference_columns.equals(df.columns):
        print("✅ Column names and order match 2024")
    else:
        print("❌ Schema mismatch")


2011
✅ Column names and order match 2024

2012
✅ Column names and order match 2024

2014
✅ Column names and order match 2024

2018
✅ Column names and order match 2024

2021
✅ Column names and order match 2024

2023
✅ Column names and order match 2024

2024
✅ Column names and order match 2024


### Data INFO

In [54]:
for year, df in datasets.items():

    print(f"\n{'='*80}")
    print(f"{year}")
    print("="*80)

    df.info()


2011
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2589121 entries, 0 to 2589120
Data columns (total 33 columns):
 #   Column                               Dtype  
---  ------                               -----  
 0   Health Service Area                  object 
 1   Hospital County                      object 
 2   Operating Certificate Number         float64
 3   Permanent Facility Id                float64
 4   Facility Name                        object 
 5   Age Group                            object 
 6   Zip Code                             object 
 7   Gender                               object 
 8   Race                                 object 
 9   Ethnicity                            object 
 10  Length of Stay                       object 
 11  Type of Admission                    object 
 12  Patient Disposition                  object 
 13  Discharge Year                       int64  
 14  CCSR Diagnosis Code                  int64  
 15  CCSR Diagnosis Description

### Missing Values

In [55]:
for year, df in datasets.items():

    print(f"\n{'='*80}")
    print(f"Missing Values - {year}")
    print("="*80)

    nulls = pd.DataFrame({
        "Null Count": df.isnull().sum(),
        "Null %": (df.isnull().mean()*100).round(2)
    })

    display(nulls[nulls["Null Count"]>0].sort_values("Null Count", ascending=False))


Missing Values - 2011


,Null Count,Null %
Payment Typology 3,1854118,71.61
Payment Typology 2,773485,29.87
Zip Code,6348,0.25
Health Service Area,4606,0.18
Hospital County,4606,0.18
Permanent Facility Id,4606,0.18
Operating Certificate Number,4606,0.18
APR Risk of Mortality,77,0.00
APR Severity of Illness Description,77,0.00



Missing Values - 2012


,Null Count,Null %
Payment Typology 3,1824441,71.70
Payment Typology 2,763177,29.99
Zip Code,38670,1.52
Health Service Area,6788,0.27
Hospital County,6788,0.27
Permanent Facility Id,6788,0.27
Operating Certificate Number,6788,0.27
APR Risk of Mortality,89,0.00
APR Severity of Illness Description,89,0.00



Missing Values - 2014


...

,Null Count,Null %
Birth Weight,1925333,90.17
Payment Typology 3,1803108,84.44
Payment Typology 2,1097001,51.38
CCSR Procedure Code,583187,27.31
CCSR Procedure Description,583187,27.31
Zip Code,40246,1.88
Operating Certificate Number,6663,0.31
Health Service Area,5214,0.24
Permanent Facility Id,5214,0.24
Hospital County,5214,0.24



Missing Values - 2023


,Null Count,Null %
Birth Weight,1921751,90.40
Payment Typology 3,1843136,86.71
Payment Typology 2,1114046,52.41
CCSR Procedure Code,611024,28.74
CCSR Procedure Description,611024,28.74
Zip Code,41883,1.97
Hospital County,5333,0.25
Health Service Area,5333,0.25
Permanent Facility Id,5333,0.25
Operating Certificate Number,5333,0.25



Missing Values - 2024


,Null Count,Null %
Birth Weight,1988915,90.54
Payment Typology 3,1956699,89.07
Payment Typology 2,1204408,54.83
CCSR Procedure Code,643764,29.31
CCSR Procedure Description,643764,29.31
Zip Code,41972,1.91
Health Service Area,5295,0.24
Operating Certificate Number,5295,0.24
Hospital County,5295,0.24
Permanent Facility Id,5295,0.24


#### Unique Values

In [56]:
for year, df in datasets.items():

    print(f"\n{'='*80}")
    print(f"Unique Values - {year}")
    print("="*80)

    display(pd.DataFrame({
        "Column": df.columns,
        "Unique Values": df.nunique()
    }))


Unique Values - 2011


,Column,Unique Values
Health Service Area,Health Service Area,8
Hospital County,Hospital County,57
Operating Certificate Number,Operating Certificate Number,189
Permanent Facility Id,Permanent Facility Id,224
Facility Name,Facility Name,225
Age Group,Age Group,5
Zip Code,Zip Code,50
Gender,Gender,3
Race,Race,4
Ethnicity,Ethnicity,3



Unique Values - 2012


,Column,Unique Values
Health Service Area,Health Service Area,8
Hospital County,Hospital County,57
Operating Certificate Number,Operating Certificate Number,188
Permanent Facility Id,Permanent Facility Id,223
Facility Name,Facility Name,239
Age Group,Age Group,5
Zip Code,Zip Code,50
Gender,Gender,3
Race,Race,4
Ethnicity,Ethnicity,3



Unique Values - 2014


...

,Column,Unique Values
Health Service Area,Health Service Area,8
Hospital County,Hospital County,57
Operating Certificate Number,Operating Certificate Number,168
Permanent Facility Id,Permanent Facility Id,204
Facility Name,Facility Name,204
Age Group,Age Group,5
Zip Code,Zip Code,50
Gender,Gender,3
Race,Race,4
Ethnicity,Ethnicity,4



Unique Values - 2023


,Column,Unique Values
Health Service Area,Health Service Area,8
Hospital County,Hospital County,57
Operating Certificate Number,Operating Certificate Number,160
Permanent Facility Id,Permanent Facility Id,207
Facility Name,Facility Name,206
Age Group,Age Group,5
Zip Code,Zip Code,50
Gender,Gender,3
Race,Race,4
Ethnicity,Ethnicity,4



Unique Values - 2024


,Column,Unique Values
Health Service Area,Health Service Area,8
Hospital County,Hospital County,57
Operating Certificate Number,Operating Certificate Number,158
Permanent Facility Id,Permanent Facility Id,205
Facility Name,Facility Name,203
Age Group,Age Group,5
Zip Code,Zip Code,50
Gender,Gender,3
Race,Race,4
Ethnicity,Ethnicity,4


#### Description

In [57]:
for year, df in datasets.items():

    print(f"\n{'='*80}")
    print(f"Describe - {year}")
    print("="*80)

    display(df.describe(include='all').T)


Describe - 2011


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Health Service Area,2584515,8,New York City,1206833,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hospital County,2584515,57,Manhattan,429648,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Operating Certificate Number,2584515.0,NaN,NaN,NaN,5006666.194054,2249886.224561,101000.0,2951001.0,5907001.0,7002002.0,7004010.0
Permanent Facility Id,2584515.0,NaN,NaN,NaN,1034.659647,654.217195,1.0,541.0,1099.0,1450.0,9250.0
Facility Name,2589121,225,Mount Sinai Hospital,57286,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age Group,2589121,5,70 or Older,716158,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip Code,2582773,50,112,355821,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,2589121,3,F,1458160,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Race,2589121,4,White,1555193,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ethnicity,2589121,3,Not Span/Hispanic,2148436,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Describe - 2012


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Health Service Area,2537755,8,New York City,1197768,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hospital County,2537755,57,Manhattan,432513,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Operating Certificate Number,2537755.0,NaN,NaN,NaN,5021651.198469,2250693.361946,101000.0,2951001.0,5907001.0,7002002.0,7004010.0
Permanent Facility Id,2537755.0,NaN,NaN,NaN,1043.646685,691.659353,1.0,541.0,1122.0,1450.0,9250.0
Facility Name,2544543,239,Mount Sinai Hospital,58722,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age Group,2544543,5,70 or Older,697658,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip Code,2505873,50,112,352179,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,2544543,3,F,1431359,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Race,2544543,4,White,1450676,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ethnicity,2544543,3,Not Span/Hispanic,2099400,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Describe - 2014


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Health Service Area,2130046,8,New York City,948517,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hospital County,2130046,57,Manhattan,366442,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Operating Certificate Number,2128597.0,NaN,NaN,NaN,4957985.457382,2261388.297997,101000.0,2951001.0,5902001.0,7002020.0,7004010.0
Permanent Facility Id,2130046.0,NaN,NaN,NaN,1027.173082,715.216373,1.0,528.0,1045.0,1453.0,10355.0
Facility Name,2135260,204,Mount Sinai Hospital,49972,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age Group,2135260,5,70 or Older,630265,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip Code,2095014,50,112,260307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,2135260,3,F,1163725,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Race,2135260,4,White,1161894,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ethnicity,2135260,4,Not Span/Hispanic,1600636,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Describe - 2023


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Health Service Area,2120421,8,New York City,963525,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hospital County,2120421,57,Manhattan,377402,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Operating Certificate Number,2120421.0,NaN,NaN,NaN,5086379.338246,2240880.801965,101000.0,2953000.0,5957001.0,7002032.0,7004010.0
Permanent Facility Id,2120421.0,NaN,NaN,NaN,1127.537116,1181.425245,1.0,541.0,1097.0,1454.0,15485.0
Facility Name,2125754,206,North Shore University Hospital,49154,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age Group,2125754,5,70 or Older,665216,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip Code,2083871,50,112,255597,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,2125754,3,F,1158896,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Race,2125754,4,White,1117779,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ethnicity,2125754,4,Not Span/Hispanic,1501423,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Describe - 2024


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Health Service Area,2191442,8,New York City,996124,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hospital County,2191442,57,New York,389375,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Operating Certificate Number,2191442.0,NaN,NaN,NaN,5087131.520291,2245914.005845,101000.0,2953000.0,5957001.0,7002032.0,7004010.0
Permanent Facility Id,2191442.0,NaN,NaN,NaN,1279.495557,1869.344069,1.0,541.0,1117.0,1456.0,15620.0
Facility Name,2196737,203,MOUNT SINAI HOSPITAL,53353,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age Group,2196737,5,70 or Older,706970,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip Code,2154765,50,112,266231,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,2196737,3,F,1195617,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Race,2196737,4,White,1147354,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ethnicity,2196737,4,Not Span/Hispanic,1540538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
